# Lab 06: output strutturato (GBNF e JSON Schema)

Lab: `corso/laboratori/lab-06-01-gbnf e lab-06-02-estrazione-json`

Esegui le celle in ordine (Runtime > Esegui tutto). Scarica fino a 1 GB e richiede alcuni minuti.


## 1. Preparazione

Scarica i binari ufficiali di llama.cpp (build `b11046`, quella del corso) e il materiale del lab.

In [ ]:
import os, subprocess
V = "b11046"
HOME = os.path.expanduser("~")
REL = f"{HOME}/llama-lab/llama-{V}"
if not os.path.exists(f"{REL}/llama"):
    subprocess.run(
        f"mkdir -p {HOME}/llama-lab && cd {HOME}/llama-lab && "
        f"curl -sLO https://github.com/ggml-org/llama.cpp/releases/download/{V}/llama-{V}-bin-ubuntu-x64.tar.gz && "
        f"tar -xzf llama-{V}-bin-ubuntu-x64.tar.gz", shell=True, check=True)
os.environ["PATH"] = REL + ":" + os.environ["PATH"]
T = min(len(os.sched_getaffinity(0)), 4)                      # thread = core disponibili (oltre i core fisici si rallenta)
os.environ["T"] = str(T)
!llama version

In [ ]:
BASE = "/content" if os.path.isdir("/content") else HOME
if not os.path.exists(f"{BASE}/corso"):
    !git clone -q --depth 1 https://github.com/JoeBosi/llama-cpp-course.git {BASE}/corso
LABS = f"{BASE}/corso/corso/laboratori"
os.environ["LABS"] = LABS

Scarica il modello di prova (785 MiB, licenza Apache 2.0) nella cache di Hugging Face.

In [ ]:
!llama download -hf ggml-org/Qwen3.5-0.8B-GGUF 2>&1 | tail -3
!llama cli -cl

## 2. Grammatica GBNF (lab 06-01)

Una grammatica elenca le sole stringhe che il modello può produrre.

In [ ]:
%%bash
cat $LABS/lab-06-01-gbnf/domotica.gbnf

In [ ]:
%%bash
cd $LABS/lab-06-01-gbnf
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -rea off -t $T -n 40 --temp 0 --grammar-file domotica.gbnf \
  -sys "Traduci la richiesta in comandi domotici, uno per riga, forma: 'accendi|spegni <luce|presa|tapparella|ventola> <cucina|soggiorno|camera|bagno|ingresso>' oppure 'imposta termostato <stanza> <gradi>'. Solo i comandi necessari." \
  -p "Sono in cucina, fa buio e ho freddo in soggiorno: voglio 22 gradi." < /dev/null 2>&1 | sed -n '/^> /,/Prompt:/p'

## 3. Estrazione JSON con schema (lab 06-02)

Il server traduce uno JSON Schema in grammatica: il risultato è sempre JSON valido, con i tipi giusti.

In [ ]:
import subprocess, time, urllib.request

def avvia_server(args, porta=8080, attesa=300):
    """Avvia `llama serve` in background e aspetta che risponda su /health."""
    log = open("/tmp/llama-server.log", "w")
    p = subprocess.Popen(["llama", "serve", *args.split(), "--host", "127.0.0.1", "--port", str(porta)],
                         stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(attesa):
        try:
            if urllib.request.urlopen(f"http://127.0.0.1:{porta}/health", timeout=2).status == 200:
                return p
        except Exception:
            time.sleep(1)
    raise RuntimeError("il server non risponde: " + open("/tmp/llama-server.log").read()[-1500:])

def ferma_server(p):
    p.terminate(); p.wait()

In [ ]:
p = avvia_server("-hf ggml-org/Qwen3.5-0.8B-GGUF -c 2048 -np 1 -a qwen-0.8b -t %d" % T)

In [ ]:
%%bash
cd $LABS/lab-06-02-estrazione-json
python3 estrai.py

In [ ]:
ferma_server(p)